In [2]:
import json
import os
import time

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import transforms

from dataset_class import MultiTaskObjectDetectionDataset, collate_fn
from multi_task_model import MultiTaskModel

def compute_iou_torch(boxes1, boxes2):
    area1 = (boxes1[:, 2] - boxes1[:, 0]) * (boxes1[:, 3] - boxes1[:, 1])
    area2 = (boxes2[:, 2] - boxes2[:, 0]) * (boxes2[:, 3] - boxes2[:, 1])
    lt = torch.max(boxes1[:, None, :2], boxes2[:, :2])
    rb = torch.min(boxes1[:, None, 2:], boxes2[:, 2:])
    wh = (rb - lt).clamp(min=0)
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = area1[:, None] + area2 - inter
    iou = inter / union
    return iou


def evaluate(model, loader, device, model_type="default", iou_threshold=0.5, confidence_threshold=0.6):
    start_time = time.time()  # ⏱️ início

    model.eval()
    total = 0
    correct_weather, correct_scene, correct_time = 0, 0, 0
    total_gt = 0
    correct_category = 0
    detections_count = 0

    with torch.no_grad(), torch.cuda.amp.autocast():
        for batch_idx, (images, targets, attrs) in enumerate(loader):
            if not images:
                continue
            images = [img.to(device) for img in images]

            outputs = model(images)
            detections = outputs if isinstance(outputs, list) else model.detection_model(images)
            w_logits, s_logits, t_logits = model.fc_weather, model.fc_scene, model.fc_timeofday

            feats = model.backbone(torch.stack(images))
            pooled = model.attr_pool(feats).view(len(images), -1)
            w_preds = torch.argmax(w_logits(pooled), dim=1).to(device)
            s_preds = torch.argmax(s_logits(pooled), dim=1).to(device)
            t_preds = torch.argmax(t_logits(pooled), dim=1).to(device)

            w_true = torch.stack([a["weather"] for a in attrs]).to(device)
            s_true = torch.stack([a["scene"] for a in attrs]).to(device)
            t_true = torch.stack([a["timeofday"] for a in attrs]).to(device)

            correct_weather += (w_preds == w_true).sum().item()
            correct_scene += (s_preds == s_true).sum().item()
            correct_time += (t_preds == t_true).sum().item()
            total += len(images)

            for i in range(len(images)):
                gt_boxes = targets[i]["boxes"].to(device)
                gt_labels = targets[i]["labels"].to(device)
                total_gt += len(gt_boxes)

                pred_boxes = detections[i]["boxes"].to(device)
                pred_labels = detections[i]["labels"].to(device)
                pred_scores = detections[i]["scores"].to(device)

                keep = pred_scores >= confidence_threshold
                pred_boxes = pred_boxes[keep]
                pred_labels = pred_labels[keep]

                if len(gt_boxes) == 0 or len(pred_boxes) == 0:
                    continue

                ious = compute_iou_torch(gt_boxes, pred_boxes)
                max_iou, max_idx = ious.max(dim=1)
                matches = max_iou >= iou_threshold
                detections_count += matches.sum().item()
                correct_category += (pred_labels[max_idx[matches]] == gt_labels[matches]).sum().item()

            if batch_idx % 10 == 0:
                print(f"🔍 Avaliados {batch_idx + 1}/{len(loader)} batches...")

    weather_acc = correct_weather / total if total > 0 else 0
    scene_acc = correct_scene / total if total > 0 else 0
    time_acc = correct_time / total if total > 0 else 0
    detection_cat_acc = correct_category / total_gt if total_gt > 0 else 0
    avg_detections = detections_count / total if total > 0 else 0

    total_time = time.time() - start_time  # ⏱️ fim
    print(f"\n⏱️ Tempo total de avaliação: {total_time:.2f} segundos")

    print(f"\n📊 Avaliação em {total} imagens:")
    print(f"🌤️ Acurácia Weather: {weather_acc:.2%}")
    print(f"🌆 Acurácia Scene:   {scene_acc:.2%}")
    print(f"🌙 Acurácia Time:    {time_acc:.2%}")
    print(f"📦 Média de detecções por imagem (matched): {avg_detections:.2f}")
    print(f"🎯 Acurácia de categorias: {detection_cat_acc:.2%}")

    os.makedirs("logs", exist_ok=True)
    evaluate_path = f"logs/evaluate_log_{model_type}.json"
    evaluation = {
        "images": total,
        "accuracy_weather": round(weather_acc, 4),
        "accuracy_scene": round(scene_acc, 4),
        "accuracy_time": round(time_acc, 4),
        "average_detections_per_image": round(avg_detections, 2),
        "accuracy_categories": round(detection_cat_acc, 4),
        "evaluation_time_seconds": round(total_time, 2)
    }
    with open(evaluate_path, "w") as f:
        json.dump(evaluation, f, indent=4)
    print(f"📄 Avaliação salva em: {evaluate_path}")

    return weather_acc, scene_acc, time_acc, avg_detections, detection_cat_acc

model_path = "multi_task_model_resnet.pth"
model_type = "resnet"
data_dir = "images/val"
label_dir = "labels/val"
batch_size = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"📦 Carregando modelo salvo de: {model_path}")
model = torch.load(model_path, map_location=device, weights_only=False)
model = model.to(device)

transform = transforms.Compose([transforms.ToTensor()])
dataset = MultiTaskObjectDetectionDataset(data_dir, label_dir, transform)
subset_size = int(len(dataset) * 0.5)
indices = np.random.choice(len(dataset), subset_size, replace=False)
subset = Subset(dataset, indices)
loader = DataLoader(subset, batch_size=batch_size, collate_fn=collate_fn)

print("🚀 Iniciando avaliação...")
evaluate(model, loader, device, model_type)

📦 Carregando modelo salvo de: multi_task_model_resnet.pth
Todas as imagens possuem JSON correspondente.
🚀 Iniciando avaliação...


C:\Users\ctw02813\AppData\Local\Temp\ipykernel_20496\1268031448.py:36: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.no_grad(), torch.cuda.amp.autocast():


🔍 Avaliados 1/63 batches...
🔍 Avaliados 11/63 batches...
🔍 Avaliados 21/63 batches...
🔍 Avaliados 31/63 batches...
🔍 Avaliados 41/63 batches...
🔍 Avaliados 51/63 batches...
🔍 Avaliados 61/63 batches...

⏱️ Tempo total de avaliação: 509.90 segundos

📊 Avaliação em 499 imagens:
🌤️ Acurácia Weather: 42.48%
🌆 Acurácia Scene:   40.48%
🌙 Acurácia Time:    54.51%
📦 Média de detecções por imagem (matched): 1.86
🎯 Acurácia de categorias: 12.18%
📄 Avaliação salva em: logs/evaluate_log_resnet.json


(0.4248496993987976,
 0.40480961923847697,
 0.5450901803607214,
 1.8637274549098197,
 0.12179381102101605)